<a href="https://colab.research.google.com/github/Sneha17A/ML_Projects/blob/main/HandBag_Shoes_Classifier_CNN.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Building Handbags-Shoes Classifier with Convolutional neural network

In [ ]:
import os
import json
import math
import numpy as np
import matplotlib.pyplot as plt
import keras
from keras import losses
from keras import ops
from keras import optimizers
from keras.optimizers import schedules
from keras import metrics
from keras.applications.imagenet_utils import decode_predictions
import keras_hub
import tensorflow as tf
import tensorflow_datasets as tfds
from tensorflow import keras
from keras.layers import Lambda
import pandas as pd

In [ ]:
!!pip install -q git+https://github.com/keras-team/keras-hub.git
!!pip install -q --upgrade keras

In [ ]:
keras.utils.set_random_seed(42)

In [ ]:
!wget -q -P ./ https://www.dropbox.com/s/w07liww46kgxo1m/handbags-shoes.zip
!unzip -qq handbags-shoes.zip

Data Preprocessing

MNIST has 60K training examples.
now with 200 images..

In [ ]:
import os, shutil, pathlib
base_dir = pathlib.Path("/content/handbags-shoes")

for category in ('handbags','shoes'):
  fnames = os.listdir(base_dir / category)
  dir = base_dir/'train'/category
  os.makedirs(dir)
  #first 50 example for training set
  for fname in fnames[:50]:
    shutil.copyfile(src=base_dir/category/fname, dst=dir/fname)
  dir = base_dir/'validation'/category
  os.makedirs(dir)
  #next 25 examples for validation test
  for fname in fnames[50:70]:
    shutil.copyfile(src=base_dir/category/fname, dst=dir/fname)
  dir = base_dir/'test'/category
  os.makedirs(dir)
  #remaining examples for test set
  for fname in fnames[75:]:
    shutil.copyfile(src=base_dir/category/fname, dst=dir/fname)

It creates directory struture

In [ ]:
!find /content/handbags-shoes -type d

It is JPEG data. it has to convert to tensors. resize them to a standard size. group them into batches. keras has a single function to do all these actions

In [ ]:
train_dataset = keras.utils.image_dataset_from_directory(
    base_dir /'train',
    image_size=(224,224),
    batch_size=32
)
validation_dataset = keras.utils.image_dataset_from_directory(
    base_dir /'validation',
    image_size=(224,224),
    batch_size=32
)
test_dataset = keras.utils.image_dataset_from_directory(
    base_dir /'test',
    image_size=(224,224),
    batch_size=32
)

In [ ]:
for images,_ in train_dataset.take(1):
  print(images[0].shape)

In [ ]:
num_classes = 2
def preprocess_data(image,label):
  label = tf.one_hot(label,num_classes)
  return image, label
train_dataset = train_dataset.map(preprocess_data)
validate_dataset = validation_dataset.map(preprocess_data)
test_dataset = test_dataset.map(preprocess_data)
for images,labels in train_dataset.take(1):
  print("Image shape:", images.shape)
  print("Label shape:", labels.shape)

In [ ]:
#few examples

plt.figure(figsize=(10,10))
for images, labels in train_dataset.take(1):
  for i in range(9):
    ax = plt.subplot(3,3,i+1)
    plt.imshow(images[i].numpy().astype("uint8"))
    plt.title(int(tf.argmax(labels[i]==1)))
    plt.axis("off")

In [ ]:
patch_size = 16
image_size = 224

In [ ]:
def get_patches(images,patch_size=16):
  input_shape = tf.shape(images)
  batch_size = input_shape[0]
  height = input_shape[1]
  width = input_shape[2]
  channels = input_shape[3]
  num_patches_h = height//patch_size
  num_patches_w = width//patch_size
  patches = tf.image.extract_patches(images=images,
                                   sizes=[1,patch_size,patch_size,1],
                                   strides=[1,patch_size,patch_size,1],
                                   rates=[1,1,1,1],
                                   padding='VALID')
  patches = tf.reshape(tf.cast(patches,dtype=tf.float32),
                       (
                           batch_size,
                           num_patches_h*num_patches_w,
                           patch_size*patch_size*channels
                       ),
                       )
  return patches

In [ ]:
for images, _ in train_dataset.take(1):
  plt.figure(figsize=(4,4))
  resized_image = keras.ops.image.resize(
      keras.ops.convert_to_tensor([images[0]]),
      (image_size,image_size)
  )
  no_channels = keras.ops.shape(resized_image)[-1]
  plt.imshow(images[0].numpy().astype("uint8"))
  plt.axis("off")

In [ ]:
patches = get_patches(resized_image)
print(f"Image size: {image_size} X {image_size}")
print(f"Patch size: {patch_size} X {patch_size}")
print(f"Patches per image: {patches.shape[1]}")
print(f"Elements per patch: {patches.shape[-1]}")
n = int(np.sqrt(patches.shape[1]))
plt.figure(figsize=(4,4))
for i, patch in enumerate(patches[0]):
  ax = plt.subplot(n,n,i+1)
  patch_img = keras.ops.reshape(patch.numpy(),(16,16,no_channels))
  plt.imshow(keras.ops.convert_to_numpy(patch_img).astype("uint8"))
  plt.axis("off")

In [ ]:
num_patches = (image_size//patch_size)**2
projection_dim = 64
num_heads = 8
transformer_units = [
    projection_dim*2,
    projection_dim,
]
num_transformer_layers = 2
mlp_head_units=[
    512,
    256,
]
num_classes = 2
input_shape = (image_size,image_size,3)

In [ ]:
inputs = keras.Input(shape = input_shape)
data_augmentation = keras.Sequential(
    [
        keras.layers.RandomFlip("horizontal"),
        keras.layers.RandomRotation(0.1),
        keras.layers.RandomZoom(0.1),
        keras.layers.RandomTranlation(0.1,0.1),
    ])
x = data_augmentation(inputs)
patches = Lambda(get_patches,output_shape=(num_patches,patch_size*patch_size*3))(x)
projection = layers.Dense(units=projection_dim)
position_embedding = layers.Embedding(
    input_dim=num_patches,output_dim=projection_dim
)
encoded_patches = projection(patches) + position_embedding(keras.ops.expand_dim(keras.ops.arange(start=0,stop=num_patches,step=1),
                                                                                axis=0))
for _ in range(num_transformer_layers):
  x1 = layers.LayerNormalization(epsilon=1e-6)(encoded_patches)
  attention_output = layers.MultiHeadAttention(
      num_heads=num_heads,key_dim=projection_dim,dropout=0.1
  )(x1,x1)
  x2 = layers.Add()([attention_output, encoded_patches])
  x3 = layers.LayerNormalization(epsilon=1e-6)(x2)
  for units in transformer_units:
    x3 = layers.Dense(units,activation = keras.activations.gelu)(x3)
    x3 = layers.Dropout(0.5)(x3)
  encoded_patches = layers.Add()([x3,x2])
representation = layers.LayerNormalization(epsilon=1e-6)(encoded_patches)
representation = layers.Flatten()(representation)
representation = layers.Dropout(0.5)(representation)
for units in mlp_head_units:
  representation = layers.Dense(units,activation=keras.activations.gelu)(representation)
  representation = layers.Dropout(0.5)(representation)
logits = layers.Dense(num_classes)(representation)
model = keras.Model(inputs=inputs,outputs=logits)
